# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Kwizera Mugwaneza Frank
**Student ID:** 36432028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
# Chose GROQ

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [18]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from dotenv import load_dotenv

load_dotenv()  # reads a local .env; harmless no-op on Colab
API_KEY = os.environ.get("GROQ_API_KEY")

if API_KEY is None:  # fall back to the Colab Secrets panel
    from google.colab import userdata

    API_KEY = userdata.get("GROQ_API_KEY")

assert API_KEY, (
    "No API key found. Put GROQ_API_KEY=... in a .env file (and .gitignore it)."
)
print(
    "API key loaded from the environment, length:", len(API_KEY)
)  # never print the key

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "qwen/qwen3.6-27b"

print("Client ready.")

API key loaded from the environment, length: 56
Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [19]:
import time


def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
    return_usage=False,
):
    """One-shot chat completion. Returns the reply text, or (text, usage)."""
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            break
        except Exception as err:
            if "rate" not in str(err).lower() or attempt == 4:
                raise
            wait = 2**attempt
            print(f"  rate limited, retrying in {wait}s ...")
            time.sleep(wait)

    text = response.choices[0].message.content
    return (text, response.usage) if return_usage else text


# One simple question, so we can see the shape of a real response.
answer, usage = ask_llm(
    "In two sentences, explain what a susu savings scheme is in Ghana.",
    system_prompt="You are a concise explainer of financial services in West Africa.",
    temperature=0.7,
    return_usage=True,
)
print(answer)

# Token accounting:
print("\nresponse.usage:", usage)
print(f"prompt_tokens     = {usage.prompt_tokens}")
print(f"completion_tokens = {usage.completion_tokens}")
print(f"total_tokens      = {usage.total_tokens}")


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Topic:** Susu savings scheme in Ghana
   - **Format:** Exactly two sentences
   - **Role/Context:** Concise explainer of financial services in West Africa

2.  **Identify Key Concepts of Susu in Ghana:**
   - Traditional/informal savings system
   - Operated by local collectors (susu collectors)
   - Involves regular small cash contributions from participants
   - Collectors pool funds and often provide small loans or return the accumulated savings at the end of a cycle
   - Serves unbanked/low-income populations
   - Rooted in West African financial culture

3.  **Draft - Sentence 1 (Definition & Mechanism):**
   A susu is a traditional, informal savings system in Ghana where collectors gather small, regular cash contributions from individuals, typically through weekly or monthly visits.

4.  **Draft - Sentence 2 (Purpose/Outcome & Context):**
   The pooled funds are either returned to participants at the end of a

**Student Reasoning — Anatomy of a call**

*1. What is the difference between the `system` and `user` roles? Give an example of something that belongs in each.*
> The `system` message carries the standing context for the whole conversation: who the model is,
what rules it must obey, and what shape its output must take. 
> - **system:** "You are an assistant to a microfinance loan officer in Ghana. Be factual, never
  invent details, and answer in 3-4 sentences."


> The `user` message carries the
specific request and data for this one turn. A useful rule of thumb for this lab: anything that
would be *identical* for all six letters belongs in `system`; anything that *changes per letter*
belongs in `user`.
> - **user:** the text of letter L001, prefixed with "Summarize this loan application:".

*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> A token is the atomic unit a model reads.

> Providers bill per token because cost tracks tokens, not requests. Each Request can have varying number of words, or even documents which has more tokens that a prompt.
For example in this run the prompt_tokens = 63 tokens and it the  completion_tokens = 85 tokens which brought the  total_tokens = 148 tokens. So charging per request would be a bad business model.

### Part 1.2 — Temperature: the randomness dial

In [20]:
TEMP_QUESTION = "Suggest a name for a savings product for market traders in Accra."


# RUN 1: Temperature 0.0
print("temperature = 0.0")


answers_0 = []
for i in range(5):
    # call the API
    answer = ask_llm(TEMP_QUESTION, temperature=0.0, max_tokens=60).strip()
    answers_0.append(answer)
    print(f"[{i + 1}] {answer}\n")

# Count distinct answers
distinct_count_0 = len(set(answers_0))
print(f"{distinct_count_0} distinct answer(s) out of 5 runs\n")


# RUN 2: Temperature 1.2
print("temperature = 1.2")


answers_1 = []
for i in range(5):
    # call the API
    answer = ask_llm(TEMP_QUESTION, temperature=1.2, max_tokens=60).strip()
    answers_1.append(answer)
    print(f"[{i + 1}] {answer}\n")

# Count distinct answers
distinct_count_1 = len(set(answers_1))
print(f"{distinct_count_1} distinct answer(s) out of 5 runs\n")

temperature = 0.0
[1] <think>
Here's a thinking process:

1.  **Analyze User Request:**
   - **Product Type:** Savings product
   - **Target Audience:** Market traders
   - **Location:** Accra, Ghana
   - **Goal:** Suggest a name

2.  **Key

[2] <think>
Here's a thinking process:

1.  **Analyze User Request:**
   - **Product Type:** Savings product
   - **Target Audience:** Market traders
   - **Location:** Accra, Ghana
   - **Goal:** Suggest a name

2.  **Ident

[3] <think>
Here's a thinking process:

1.  **Analyze User Request:**
   - **Product Type:** Savings product
   - **Target Audience:** Market traders
   - **Location:** Accra, Ghana
   - **Goal:** Suggest a name

2.  **Ident

[4] <think>
Here's a thinking process:

1.  **Analyze User Request:**
   - **Product Type:** Savings product
   - **Target Audience:** Market traders
   - **Location:** Accra, Ghana
   - **Goal:** Suggest a name

2.  **Ident

[5] <think>
Here's a thinking process:

1.  **Analyze User Request:**
   - **Pro

**Student Reasoning — Temperature**

*What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?*

> I observerd at temperature=0.0 there was only one distinct answer but at temperature=1.2, there were 4/5 distinct answers. To reproduce a consistent answer, 0 is the appropriate temperature. 

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [21]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter}"

v1_summaries = {}
for lid in ["L002", "L006"]:
    v1_summaries[lid] = ask_llm(
        SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]), temperature=0.0
    )

    print(f"{lid} SUMMARY V1 (naive)")
    print(v1_summaries[lid], "\n")

# Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. You write short, "
    "factual briefs of loan application letters so a busy officer can scan a file.\n"
    "Rules you must follow:\n"
    "- Exactly 3 to 4 sentences. Plain prose. No bullet points, no headings, no preamble.\n"
    "- Use ONLY facts stated in the letter. Never invent names, amounts, dates, income, "
    "collateral, business registration or repayment terms.\n"
    "- If the letter does not state an important fact (amount, income, collateral or "
    "guarantor, repayment term), say it is not stated rather than guessing.\n"
    '- Attribute the applicant\'s claims to the applicant ("the applicant states..."), '
    "do not present them as verified fact.\n"
    "- Neutral tone. No praise, no sympathy, no judgement, and never say whether the loan "
    "should be approved or rejected.\n"
    "- Reproduce all amounts in GHS exactly as written in the letter."
)
SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

v2_summaries = {}
for lid in ["L002", "L006"]:
    v2_summaries[lid] = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=LETTERS[lid]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0,
    )

# Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for lid in ["L002", "L006"]:
    print(f"{lid} V1 vs V2")
    print("ORIGINAL LETTER")
    print(LETTERS[lid])
    print("\nV1 (naive)")
    print(v1_summaries[lid])
    print("\nV2 (role + constraints, temperature=0)")
    print(v2_summaries[lid])
    print()

L002 SUMMARY V1 (naive)

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Sender:** Kwame Boateng, commercial driver in Kumasi
   - **Request:** GHS 25,000 urgently
   - **Purpose:** Repair trotro engine, settle personal debts
   - **Repayment Plan:** "Whenever the money comes" (after festive season when business picks up)
   - **Collateral:** None
   - **Tone/Context:** Urgent, informal, relies on faith/goodwill ("God willing")
   - **Key Elements to Summarize:** Who, what, why, repayment terms, collateral status, urgency.

2.  **Identify Core Information:**
   - **Who:** Kwame Boateng (commercial driver in Kumasi)
   - **What:** Requesting GHS 25,000
   - **Why:** Urgent trotro engine repair and personal debt settlement
   - **Repayment:** Flexible/uncertain ("whenever money comes," expects improvement post-festive season)
   - **Collateral:** None
   - **Urgency:** High

3.  **Draft Summary (Mental Refinement):**
   Kwame Boateng, a commercial driver in Kumasi,

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
> V1 had no length or format. e.g "Summarize this:" still gabe a long answer, sometimes with headings or bullets. But V2 provided 3-4 sentences easily scanable with no bullet points, and headings.

> V1 carries phrases like L002's "business will surely pick up after the festive season" and L006's "full of energy" straight through as though they were established findings. V2 corrects this via the constraints "Attribute the applicant's claims to the applicant" and "Neutral tone. No praise, no sympathy, no judgement."

> V1 has a tendency to added details the letter never contained e.g. a formal repayment term for L002, who only says "whenever the money comes" A fabricated figure entering the brief becomes an input to a real credit decision. V2 strictly limits this with "Use ONLY facts stated in the letter. Never invent names, amounts, dates, income, collateral, business registration or repayment terms."

> V1 summarizes only what is present, but the officer desperately needs to know what is absent. V2 demands gap identification: "If the letter does not state an important fact ... say it is not stated rather than guessing."

>  V1 lacks context on what "important" means. Generic summarization optimizes for coverage; the officer specifically needs the requested amount, purpose, capacity to repay, and security.  The system prompt firmly grounds the LLM in its specific role: assistant to a microfinance loan officer.

*2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?*

> The consequences of inventing details are dangerous because these are fabricated and seem legitimate it might be hard later to distinguish which is which. This is Hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.